In [1]:
'''
This is the prototype of their SNN, which uses simple regression, but with 2 different models, that have 100 output neurons instead of 200. 
The idea is that SBP and DBP predictions are separate so they can use their own input features and weights to more accurately predict SBP and DBP values
To beat: MAE Label 1 (SBP) = 9.2680, MAE Label 2 (DBP) = 6.4475

To add: 
eps = 1e-8 for all 4 denominators
'''
import torch
import torch.nn as nn
import torch.optim as optim
import snntorch as snn
from snntorch import spikegen
import pandas as pd
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
from snntorch import surrogate, functional as SF
from scipy.stats import beta as beta_dist


In [ ]:
def combined_beta_cdf(vsignal):
    v = float(vsignal)
    if v < 0.5:
        vnorm = 2 * v
        return beta_dist.cdf(vnorm, a=1.0, b=0.75)
    else:
        vnorm = 2 * v - 1
        return beta_dist.cdf(vnorm, a=0.75, b=1.0)

'''
def rate_encoding_beta(X, timesteps=50):
    batch_size, N_features = X.shape
    spike_train = torch.zeros(timesteps, batch_size, N_features)
    for b in range(batch_size):
        for f in range(N_features):
            Pfr = combined_beta_cdf(X[b, f])
            spike_train[:, b, f] = torch.bernoulli(Pfr * torch.ones(timesteps))
    return spike_train
'''

def rate_encoding_beta(X, timesteps=50):
    batch_size, N_features = X.shape
    # compute all beta CDF values at once using numpy, then convert back to tensor
    X_np  = X.numpy()
    probs = np.vectorize(combined_beta_cdf)(X_np)
    probs = torch.tensor(probs, dtype=torch.float32)
    probs = probs.unsqueeze(0).expand(timesteps, -1, -1)
    return torch.bernoulli(probs)
'''
def rate_encoding_uniform(X, timesteps=50):
    batch_size, N_features = X.shape
    spike_train = torch.zeros(timesteps, batch_size, N_features)
    for b in range(batch_size):
        for f in range(N_features):
            Pfr = X[b, f]
            spike_train[:, b, f] = torch.bernoulli(Pfr * torch.ones(timesteps))
    return spike_train
'''

def rate_encoding_uniform(X, timesteps=50):
    probs = X.unsqueeze(0).expand(timesteps, -1, -1)
    return torch.bernoulli(probs)

def encode_batch(x, num_steps, method):
    '''
    x      : [batch, n_features] normalized to [0,1]
    method : 'spikegen' | 'uniform' | 'beta'
    Returns: [num_steps, batch, n_features]
    '''
    if method == 'spikegen':
        return spikegen.rate(x, num_steps=num_steps)
    elif method == 'uniform':
        return rate_encoding_uniform(x, timesteps=num_steps)
    elif method == 'beta':
        return rate_encoding_beta(x, timesteps=num_steps)
    else:
        raise ValueError(f'Unknown encoding: {method}')

In [ ]:
num_steps = 40

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

class Net(nn.Module):
    def __init__(self, num_inputs, num_hidden, num_hidden2,
                 num_outputs, beta_val):
        super().__init__()
        self.fc1  = nn.Linear(num_inputs, num_hidden)
        self.lif1 = snn.Leaky(beta=beta_val)
        self.fc2  = nn.Linear(num_hidden, num_hidden2)
        self.lif2 = snn.Leaky(beta=beta_val)
        self.fc3  = nn.Linear(num_hidden2, num_outputs)
        self.lif3 = snn.Leaky(beta=beta_val)

    def forward(self, xf):
        mem1 = self.lif1.init_leaky()
        mem2 = self.lif2.init_leaky()
        mem3 = self.lif3.init_leaky()
        spk_rec = []
        input_spikes =  spikegen.rate(xf, num_steps=num_steps).to(device)
        for t in range(num_steps):
            cur1 = self.fc1(input_spikes[t])
            spk1, mem1 = self.lif1(cur1, mem1)
            cur2 = self.fc2(spk1)
            spk2, mem2 = self.lif2(cur2, mem2)
            cur3 = self.fc3(spk2)
            spk3, mem3 = self.lif3(cur3, mem3)
            spk_rec.append(spk3)
        return torch.stack(spk_rec)

In [4]:
def load_data(fold_num):
    return pd.read_csv(f"bcg_dataset/feat_fold_{fold_num}.csv")

In [ ]:
"""
In this case the approach was to divide SP and DP in 100 intervals, from their respective min and max values.
During training and inference, the ouput layer of the net are divided in two groups, of same size, than the firing rate of this neurons are used to predict the label.
eg.
We use 200 ouput neuron since we have 100 interval/classes. 100 of them refers to the SP and 100.
Each neuron correspond to an interval value, and we use the firing rate of each neuron to do a weighted sum and predict the blood pressure.
[70, 82,...190] 100 values evenly spaced
Final prediction = [70, 82,...190] * [firing rate of each neuron]'/[sum of the firining rates]
"""

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
ENCODING_METHODS = ['spikegen', 'uniform', 'beta']


most_important_features = ["TdiaToEnd","T_a_norm","S4_norm","TSystoDiaRise", "ppg_histogram_down_2","S4","ppg4_histogram_down_5","T_e_norm","ppg3_histogram_down_8","usdc_2"]

'''
top_N_features_SBP = [
    'AUCsys_norm', 'TdiaToEnd', 'AUCdia_norm', 'T_peak_e_norm', 'ppg_histogram_down_2',
    'T_peak_a', 'SteepDiaRise', 'vpg_max_neighbor_mean_0', 'TNegSteepest_norm', 'ppg4_max_0',
    'TSystoDiaRise', 'usdc_2', 'ppg3_histogram_down_8', 'T_peak_e', 'vpg_histogram_down_0',
    'apg_histogram_down_3', 'TNegSteepest', 'ratio_apg_b', 'TdiaRise', 'ppg4_min_0',
    'ppg_max_neighbor_mean_0', 'Td', 'dsdc_12', 'T_b', 'dsdc_18', 'Tc',
    'S4_norm', 'dsdc_2', 'sdoo', 'ppg3_histogram_down_7'
]

top_N_features_DBP = [
    'dsdc_8', 'TdiaToEnd', 'ppg_histogram_down_4', 'Td', 'S4', 'vpg_histogram_down_0',
    'vpg_max_neighbor_mean_0', 'T_peak_e', 'ppg4_histogram_down_3', 'T_peak_a',
    'dsdc_9', 'ppg3_histogram_down_8', 'SteepDiaRise', 'vpg_histogram_down_4', 'S3_norm',
    'S4_norm', 'ratio_apg_b', 'TNegSteepest', 'ppg_histogram_down_5', 'ppg_histogram_down_1',
    'ratio_apg_d', 'ppg4_histogram_down_6', 'T_b', 'ppg_histogram_down_3', 'T_peak_e_norm',
    'ppg_histogram_down_2', 'T_d_norm', 'TSystoDiaRise', 'apg_histogram_down_5', 'TdiaRise'
]
'''

top_features_SBP = [
    'TdiaToEnd', 'T_a_norm', 'S4_norm', 'TSystoDiaRise', 'ppg_histogram_down_2', 'S4',
    'ppg4_histogram_down_5', 'T_e_norm', 'ppg3_histogram_down_8', 'usdc_2', 'AUCsys_norm',
    'AUCdia_norm', 'T_peak_e_norm', 'T_peak_a', 'SteepDiaRise', 'vpg_max_neighbor_mean_0',
    'TNegSteepest_norm', 'ppg4_max_0', 'T_peak_e', 'vpg_histogram_down_0', 'apg_histogram_down_3',
    'TNegSteepest', 'ratio_apg_b', 'TdiaRise', 'ppg4_min_0', 'ppg_max_neighbor_mean_0',
    'Td', 'dsdc_12', 'T_b', 'dsdc_18'
]
top_features_DBP = [
    'dsdc_8', 'TdiaToEnd', 'ppg_histogram_down_4', 'Td', 'S4', 'vpg_histogram_down_0',
    'vpg_max_neighbor_mean_0', 'T_peak_e', 'ppg4_histogram_down_3', 'T_peak_a',
    'dsdc_9', 'ppg3_histogram_down_8', 'SteepDiaRise', 'vpg_histogram_down_4', 'S3_norm',
    'S4_norm', 'ratio_apg_b', 'TNegSteepest', 'ppg_histogram_down_5', 'ppg_histogram_down_1',
    'ratio_apg_d', 'ppg4_histogram_down_6', 'T_b', 'ppg_histogram_down_3', 'T_peak_e_norm',
    'ppg_histogram_down_2', 'T_d_norm', 'TSystoDiaRise', 'apg_histogram_down_5', 'TdiaRise'
]

batch_size = 1
num_epochs = 2
num_inputs = 30
num_hidden  = 60
num_hidden2 = 200
num_outputs = 100
topk_sbp = 30
topk_dbp = 30

beta_val    = 0.95
num_folds = 5

net_sbp = Net(num_inputs, num_hidden, num_hidden2, num_outputs, beta_val).to(device)
net_dbp = Net(num_inputs, num_hidden, num_hidden2, num_outputs, beta_val).to(device)

preferred_values_sbp = torch.linspace(70, 190, 100).to(device)
preferred_values_dbp = torch.linspace(45, 101, 100).to(device)

loss_fn = nn.MSELoss()
optimizer_sbp = optim.Adam(net_sbp.parameters(), lr=5e-4)
optimizer_dbp = optim.Adam(net_dbp.parameters(), lr=5e-4)

for fold in range(5):
    print(f"Training on Fold {fold + 1}/{num_folds}")

    test_data = load_data(fold)
    train_data = pd.concat([load_data(i) for i in range(num_folds) if i!= fold])

    X_train_sbp = torch.tensor(train_data[top_features_SBP].values, dtype=torch.float32)
    X_train_dbp = torch.tensor(train_data[top_features_DBP].values, dtype=torch.float32)
    y_train = torch.tensor(train_data.iloc[:, 2:4].values, dtype=torch.float32)

    X_test_sbp = torch.tensor(test_data[top_features_SBP].values, dtype=torch.float32)
    X_test_dbp = torch.tensor(test_data[top_features_DBP].values, dtype=torch.float32)
    y_test = torch.tensor(test_data.iloc[:, 2:4].values, dtype=torch.float32)

    train_loader = DataLoader(
        TensorDataset(X_train_sbp, X_train_dbp, y_train),
        batch_size=batch_size,
        shuffle=True
    )

    test_loader = DataLoader(
        TensorDataset(X_test_sbp, X_test_dbp, y_test),
        batch_size=batch_size,
        shuffle=False
    )

    total_train_loss_sbp = 0
    total_train_loss_dbp = 0
    for epoch in range(num_epochs):
        net_sbp.train()
        net_dbp.train()
        num_batches = 0
        for x_sbp, x_dbp, y in train_loader: 
            x_sbp = x_sbp.to(device)
            x_dbp = x_dbp.to(device)
            y = y.to(device)

            optimizer_sbp.zero_grad()
            optimizer_dbp.zero_grad()

            # SBP
            spk_sbp = net_sbp(x_sbp)
            fr_sbp = spk_sbp.mean(0)
            topk = torch.topk(fr_sbp,topk_sbp,dim=1)[1]
            fr_top = torch.gather(fr_sbp,1,topk)
            pv_top = preferred_values_sbp[topk]

            pred_sbp = (fr_top*pv_top).sum(1)/fr_top.sum(1)

            # DBP
            spk_dbp = net_dbp(x_dbp)
            fr_dbp = spk_dbp.mean(0)
            topk = torch.topk(fr_dbp,topk_dbp,dim=1)[1]
            fr_top = torch.gather(fr_dbp,1,topk)
            pv_top = preferred_values_dbp[topk]

            pred_dbp = (fr_top*pv_top).sum(1)/fr_top.sum(1)

            loss_sbp = loss_fn(pred_sbp, y[:,0])
            loss_dbp = loss_fn(pred_dbp, y[:,1])

            loss_sbp.backward()
            optimizer_sbp.step()

            loss_dbp.backward()
            optimizer_dbp.step()
            total_train_loss_sbp += loss_sbp.item()
            total_train_loss_dbp += loss_dbp.item()
            num_batches += 1

    print(f"Epoch {epoch + 1}, Training Loss sbp: {total_train_loss_sbp / num_batches:.4f}")
    print(f"Epoch {epoch + 1}, Training Loss dbp: {total_train_loss_dbp / num_batches:.4f}")



Training on Fold 1/5
Epoch 2, Training Loss sbp: 279.4408
Epoch 2, Training Loss dbp: 84.6533
Training on Fold 2/5
Epoch 2, Training Loss sbp: 221.7610
Epoch 2, Training Loss dbp: 91.9719
Training on Fold 3/5
Epoch 2, Training Loss sbp: 270.1072
Epoch 2, Training Loss dbp: 85.4145
Training on Fold 4/5
Epoch 2, Training Loss sbp: 248.5412
Epoch 2, Training Loss dbp: 66.4125
Training on Fold 5/5
Epoch 2, Training Loss sbp: 241.5104
Epoch 2, Training Loss dbp: 75.8444


In [6]:
net_sbp.eval()
net_dbp.eval()
total_samples = 0
total_mae_sbp = 0
total_mae_dbp = 0

with torch.no_grad():
        
    for x_sbp, x_dbp, y in test_loader:
        x_sbp = x_sbp.to(device)
        x_dbp = x_dbp.to(device)
        y = y.to(device)

        # SBP
        firing_rates_sbp = net_sbp(x_sbp).mean(0)
        pred_sbp = (firing_rates_sbp * preferred_values_sbp).sum(1) / firing_rates_sbp.sum(1)

        # DBP
        firing_rates_dbp = net_dbp(x_dbp).mean(0)
        pred_dbp = (firing_rates_dbp * preferred_values_dbp).sum(1) / firing_rates_dbp.sum(1)

        total_mae_sbp += torch.abs(pred_sbp - y[:,0]).sum().item()
        total_mae_dbp += torch.abs(pred_dbp - y [:,1]).sum().item()
        total_samples += y.shape[0]

    print(f"MAE Label 1 (SBP) = {total_mae_sbp / total_samples:.4f}, MAE Label 2 (DBP) = {total_mae_dbp / total_samples:.4f}")

MAE Label 1 (SBP) = 9.3616, MAE Label 2 (DBP) = 5.7579


In [ ]:
'''
I am using this cell to keep track of the features and their results:

BASELINE (their single model): 
    most_important_features = ["TdiaToEnd","T_a_norm","S4_norm","TSystoDiaRise", "ppg_histogram_down_2"
    ,"S4","ppg4_histogram_down_5","T_e_norm","ppg3_histogram_down_8","usdc_2"]
    for both models

    MAE Label 1 (SBP) = 9.2680, MAE Label 2 (DBP) = 6.4475

1.  top_features_SBP = ['AUCsys_norm', 'TdiaToEnd', 'AUCdia_norm', 'T_peak_e_norm', 'ppg_histogram_down_2',
    'T_peak_a', 'SteepDiaRise', 'vpg_max_neighbor_mean_0', 'TNegSteepest_norm', 'ppg4_max_0',]
    top_features_DBP = ['dsdc_8', 'TdiaToEnd', 'ppg_histogram_down_4', 'Td', 'S4', 'vpg_histogram_down_0',
    'vpg_max_neighbor_mean_0', 'T_peak_e', 'ppg4_histogram_down_3', 'T_peak_a',]

    MAE Label 1 (SBP) = 10.1164, MAE Label 2 (DBP) = 5.1287

3.  top_features_SBP = most_important_features
    top_features_DBP = ['dsdc_8', 'TdiaToEnd', 'ppg_histogram_down_4', 'Td', 'S4', 'vpg_histogram_down_0',
    'vpg_max_neighbor_mean_0', 'T_peak_e', 'ppg4_histogram_down_3', 'T_peak_a',]
    
    MAE Label 1 (SBP) = 10.0492, MAE Label 2 (DBP) = 5.1450

4.  Top 30 features for both SBP and DBP separately using feature_importances_ function for forestRegressors

    MAE Label 1 (SBP) = 9.2318, MAE Label 2 (DBP) = 5.0097
    
5.  Top 10 features from the original architecture + the top 20 features of my ranking done using feature_importances_ 
    that are not in the original list JUST FOR SBP, DBP is the same as above (4). 

    MAE Label 1 (SBP) = 9.3616, MAE Label 2 (DBP) = 5.7579
'''

'\nI am using this cell to keep track of the features and their results:\n\nBASELINE (their single model): \n    most_important_features = ["TdiaToEnd","T_a_norm","S4_norm","TSystoDiaRise", "ppg_histogram_down_2"\n    ,"S4","ppg4_histogram_down_5","T_e_norm","ppg3_histogram_down_8","usdc_2"]\n    for both models\n\n    MAE Label 1 (SBP) = 9.2680, MAE Label 2 (DBP) = 6.4475\n\n1.  top_features_SBP = [\'AUCsys_norm\', \'TdiaToEnd\', \'AUCdia_norm\', \'T_peak_e_norm\', \'ppg_histogram_down_2\',\n    \'T_peak_a\', \'SteepDiaRise\', \'vpg_max_neighbor_mean_0\', \'TNegSteepest_norm\', \'ppg4_max_0\',]\n    top_features_DBP = [\'dsdc_8\', \'TdiaToEnd\', \'ppg_histogram_down_4\', \'Td\', \'S4\', \'vpg_histogram_down_0\',\n    \'vpg_max_neighbor_mean_0\', \'T_peak_e\', \'ppg4_histogram_down_3\', \'T_peak_a\',]\n\n    MAE Label 1 (SBP) = 10.1164, MAE Label 2 (DBP) = 5.1287\n\n3.  top_features_SBP = most_important_features\n    top_features_DBP = [\'dsdc_8\', \'TdiaToEnd\', \'ppg_histogram_down_4